In [1]:
import pandas as pd
import time
import os
import re
import sys
from IPython.display import clear_output
# import gspread

sys.path.append('./src')
from GSheetImporter import GSheetImporter
from pullprice import yfinance_sym_dic, get_live_price

# Load arguments

In [2]:
GENERATE_MARKDOWN = True
GENERATE_HTML = True

REPORT_PATH = '../TradingAssistWebapp/pages/'
# GSHEET_CREDS = "c:/users/pbara/Documents/Python/secrets/sheets-pandas-reader-193e91a08e8e.json"
GSHEET_CREDS = '/home/pbarahimi/.credentials/gsheets.json'

# Read the trades worksheet

- Requires google account service credentials to read the full sheet regardless of the filters applied in the browser
- Refer to [`gsheet_access_instructions.txt`](https://share.gemini.google/0dLETC0zvoAe) for step-by-step instructions on how to setup access

In [3]:
SHEET_ID = "1HJ9h7UEtUQCXNA58UkZyPsHogJWBAcB1lNWt9nOPMR4"
SHEET_NAME = 'Trades'
num_cols = ['Open Price', 'Close Price', 'Commission','Risk ($)', 'Balance at Open', 'PnL']

gsheet = GSheetImporter(sheet_id=SHEET_ID, sheet_name=SHEET_NAME, credentials_path=GSHEET_CREDS)
gsheet.get_dataframe()
gsheet.to_num(num_cols)

# Keep open trades
df = gsheet.df[gsheet.df['Is Closed']==0].copy()

# Add volume weighted entry price
total_vol = df.groupby(['Account','Symbol'], as_index=False).agg({'Volume': sum})
total_vol.rename(columns={'Volume': 'Total Volume'}, inplace=True)
df = pd.merge(df, total_vol, on=['Account','Symbol'])
df['Volume Weighted Open Price'] = df['Open Price'] * (df['Volume']/df['Total Volume'])

df.head()

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Potential Profit,PnL,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Strategy,Total Volume,Volume Weighted Open Price
0,9/3/2026 10:11:59,09/03/26,Paper Trading #2,TSLA,-24.0,371.81,377.94,0.00,100608.20,411.98,...,2444.88,-147.12,0,,#VALUE!,"Trade 1, 1h,",,,-24.0,371.810000
1,8/24/2026 0:00:00,08/24/26,Tradestation - Equity,CVNA,-140.0,72.32,64.18,-2.91,42336.39,81.13,...,2250.29,1137.29,0,,#VALUE!,Link,,,-165.0,61.362424
2,9/2/2026 13:35:00,09/02/26,Tradestation - Equity,CVNA,-25.0,74.81,64.18,0.00,48760.00,76.8,...,170.25,265.75,0,,#VALUE!,,,,-165.0,11.334848
3,9/6/2026 10:00:00,06/13/26,Tradestation - Futures,ADA,177000.0,0.17,NaN,0.00,218000.00,,...,500290.5,0.00,0,,#VALUE!,,,,177000.0,0.170000
4,8/16/2026 0:00:00,08/16/26,Yvonne's Robinhood,ADA,60000.0,0.18,NaN,0.00,218000.00,,...,169192.71,0.00,0,,#VALUE!,,,,60000.0,0.180000


# Get Point Values

In [4]:
# Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Symbols'

gsheet = GSheetImporter(SHEET_ID, SHEET_NAME, GSHEET_CREDS)
all_values = gsheet.get_all_values()
point_val_df = pd.DataFrame(all_values, columns=['Symbol', 'Point Value'])
point_val_df['Point Value'] = point_val_df['Point Value'].astype(float)

point_val_df.head()

,Symbol,Point Value
0,ADA,1.0
1,BTC,1.0
2,COF,1.0
3,CVNA,1.0
4,ETH,1.0


# Get Prices

In [5]:
price_df = pd.DataFrame(df['Symbol']).drop_duplicates()
price_df['Current Price'] = price_df.Symbol.apply(lambda x : get_live_price(x, yfinance_sym_dic))
price_df

,Symbol,Current Price
0,TSLA,377.940002
1,CVNA,64.180000
3,ADA,0.249050
6,CPER,40.650002


# Append Price to trades DF

In [6]:
df = pd.merge(df, price_df, on='Symbol', how='left')
df = pd.merge(df, point_val_df, on='Symbol', how='left')
df['Point Value'] = df['Point Value'].fillna(1)
df['PnL'] = (df['Volume'] * (df['Current Price']-df['Open Price']) * df['Point Value']).round(2)
df

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Strategy,Total Volume,Volume Weighted Open Price,Current Price,Point Value
0,9/3/2026 10:11:59,09/03/26,Paper Trading #2,TSLA,-24.0,371.81,377.94,0.00,100608.20,411.98,...,0,,#VALUE!,"Trade 1, 1h,",,,-24.0,371.810000,377.940002,1.0
1,8/24/2026 0:00:00,08/24/26,Tradestation - Equity,CVNA,-140.0,72.32,64.18,-2.91,42336.39,81.13,...,0,,#VALUE!,Link,,,-165.0,61.362424,64.180000,1.0
2,9/2/2026 13:35:00,09/02/26,Tradestation - Equity,CVNA,-25.0,74.81,64.18,0.00,48760.00,76.8,...,0,,#VALUE!,,,,-165.0,11.334848,64.180000,1.0
3,9/6/2026 10:00:00,06/13/26,Tradestation - Futures,ADA,177000.0,0.17,NaN,0.00,218000.00,,...,0,,#VALUE!,,,,177000.0,0.170000,0.249050,1.0
4,8/16/2026 0:00:00,08/16/26,Yvonne's Robinhood,ADA,60000.0,0.18,NaN,0.00,218000.00,,...,0,,#VALUE!,,,,60000.0,0.180000,0.249050,1.0
5,9/3/2026 9:59:40,09/03/26,Paper Trading #1,TSLA,-14.0,371.81,377.94,0.00,61600.00,411.98,...,0,,#VALUE!,"Trade 1, 1h,",,Fibo,-14.0,371.810000,377.940002,1.0
6,9/21/2026 10:06:01,09/21/26,Tradestation - Equity,CPER,-120.0,40.69,40.65,0.00,30200.00,41.55,...,0,,#VALUE!,Link,,,-120.0,40.690000,40.650002,1.0
7,9/24/2026 10:29:02,09/24/26,Tradestation - Equity,TSLA,-8.0,378.94,377.94,0.00,30800.00,387.29,...,0,,#VALUE!,Link,,,-8.0,378.940000,377.940002,1.0


# Group by account and symbol to report

In [7]:
spacer_line = '\n\n' + 50 * '-' + '\n'

_t = df.groupby(['Account','Symbol']).agg({'Volume': sum,                                            
                                            'Volume Weighted Open Price': sum,
                                            'Current Price': 'mean',
                                            'PnL': sum,})
_t.rename(columns={'Volume Weighted Open Price': 'Open Price'}, inplace=True)
cols = ['Open Price', 'Current Price']
_t[cols] = _t[cols].round(2)


out = _t.to_string() + spacer_line
out += df.groupby('Account').agg({'PnL': sum}).to_string() + spacer_line
out += df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
out += df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
print(out)

                                 Volume  Open Price  Current Price       PnL
Account                Symbol                                               
Paper Trading #1       TSLA       -14.0      371.81         377.94    -85.82
Paper Trading #2       TSLA       -24.0      371.81         377.94   -147.12
Tradestation - Equity  CPER      -120.0       40.69          40.65      4.80
                       CVNA      -165.0       72.70          64.18   1405.35
                       TSLA        -8.0      378.94         377.94      8.00
Tradestation - Futures ADA     177000.0        0.17           0.25  13991.85
Yvonne's Robinhood     ADA      60000.0        0.18           0.25   4143.00

--------------------------------------------------
                             PnL
Account                         
Paper Trading #1          -85.82
Paper Trading #2         -147.12
Tradestation - Equity    1418.15
Tradestation - Futures  13991.85
Yvonne's Robinhood       4143.00

-----------------------

In [8]:
'''
spacer_line = '\n\n<br>\n\n' 

out = df.groupby('Account').agg({'PnL': sum}).to_markdown() + spacer_line
out += df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
out += df.groupby(['Symbol','Account'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
out += df.groupby(['Account','Symbol'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
'''
out = _t.to_markdown()
print(out)

|                                   |   Volume |   Open Price |   Current Price |      PnL |
|:----------------------------------|---------:|-------------:|----------------:|---------:|
| ('Paper Trading #1', 'TSLA')      |      -14 |       371.81 |          377.94 |   -85.82 |
| ('Paper Trading #2', 'TSLA')      |      -24 |       371.81 |          377.94 |  -147.12 |
| ('Tradestation - Equity', 'CPER') |     -120 |        40.69 |           40.65 |     4.8  |
| ('Tradestation - Equity', 'CVNA') |     -165 |        72.7  |           64.18 |  1405.35 |
| ('Tradestation - Equity', 'TSLA') |       -8 |       378.94 |          377.94 |     8    |
| ('Tradestation - Futures', 'ADA') |   177000 |         0.17 |            0.25 | 13991.9  |
| ("Yvonne's Robinhood", 'ADA')     |    60000 |         0.18 |            0.25 |  4143    |


In [9]:
if GENERATE_MARKDOWN:
    '''
    page_nm = 'acct_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:  # Save to a file
        f.write(df.groupby('Account').agg({'PnL': sum}).to_markdown())
        
    page_nm = 'sym_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_markdown())
    
    page_nm = 'sym_acct_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby(['Symbol','Account'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown())
    '''
    page_nm = 'acct_sym_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(_t.to_markdown())

In [10]:
if GENERATE_HTML:
    '''
    page_nm = 'acct_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:  # Save to a file
        t = df.groupby('Account').agg({'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
        
    page_nm = 'sym_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
    
    page_nm = 'sym_acct_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
    '''
    page_nm = 'acct_sym_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(_t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))